In [ ]:
# -*- coding: utf-8 -*-
"""CB_COMPLETO"""

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd

In [ ]:
df = pd.read_csv('/content/drive/My Drive/DATASET/DATASET_COMPLETO_230.csv',
                    encoding='utf-8-sig', quotechar='"', on_bad_lines='skip', dtype={'id': str})

In [ ]:
df.head()

,id,url,text,source,createdat,author_username,author_name,author_location,author_createdAt,author_favouritesCount,...,place_coordinates,place_type,place_country,place_full_name,place_name,place_id,place_place_type,Unnamed: 19,Unnamed: 20,Unnamed: 21
0,1000082083853713409,https://x.com/HeyJC_/status/1000082083853713409,Yo ya ni me esfuerzo en hacerlo porque luego p...,Twitter for Android,Fri May 25 18:32:04 +0000 2018,HeyJC_,Juan Carlos⚡️,"Querétaro Arteaga, México",Sat Mar 24 02:43:40 +0000 2012,23463,...,"[[[-100.5963431,20.5031701],[-100.5963431,20.9...",Polygon,Mexico,"Querétaro, Querétaro Arteaga",Querétaro,d3492c68c5661201,city,NaN,NaN,NaN
1,1000166664892174336,https://x.com/MuErTe_ChIqUiTa/status/100016666...,"@fernandac710 Además, ya obligaron a los demás...",Twitter for iPhone,Sat May 26 00:08:09 +0000 2018,MuErTe_ChIqUiTa,MuErTe cHiQuItA,México,Sun Sep 06 00:13:48 +0000 2009,5091,...,"[[[-99.1919955,19.357102],[-99.1919955,19.4041...",Polygon,Mexico,"Benito Juárez, Distrito Federal",Benito Juárez,7d93122509633720,city,NaN,NaN,NaN
2,1000178687793336320,https://x.com/Rosalinda_F_H/status/10001786877...,L@s invito a que nos sumemos al #DiaNaranja. ...,Twitter for iPhone,Sat May 26 00:55:56 +0000 2018,Rosalinda_F_H,Rosalinda Figueroa,"Santa María Huatulco, Oaxaca",Mon Apr 19 17:18:34 +0000 2010,12904,...,"[[[-96.531534,15.6574918],[-96.531534,15.92064...",Polygon,Mexico,"San Pedro Pochutla, Oaxaca",San Pedro Pochutla,f6aa0dce95d47c0d,city,NaN,NaN,NaN
3,1000185709586731008,https://x.com/ashitakasalto/status/10001857095...,@Tania_Tagle Una vez estaba en una playa nudis...,Twitter for Android,Sat May 26 01:23:50 +0000 2018,ashitakasalto,Ashitaka,NaN,Mon Jun 04 19:39:44 +0000 2012,124710,...,"[[[-118.4038571,14.5319181],[-118.4038571,32.7...",Polygon,Mexico,Mexico,Mexico,25530ba03b7d90c6,country,NaN,NaN,NaN
4,1000205783001387008,https://x.com/MuErTe_ChIqUiTa/status/100020578...,"@garobiela @fernandac710 Son tan básicos, que ...",Twitter for iPhone,Sat May 26 02:43:36 +0000 2018,MuErTe_ChIqUiTa,MuErTe cHiQuItA,México,Sun Sep 06 00:13:48 +0000 2009,5091,...,"[[[-99.1919955,19.357102],[-99.1919955,19.4041...",Polygon,Mexico,"Benito Juárez, Distrito Federal",Benito Juárez,7d93122509633720,city,NaN,NaN,NaN


In [ ]:
# Reparar fechas
df['createdat'] = pd.to_datetime(df['createdat'], errors='coerce', utc=True)
df['author_createdAt'] = pd.to_datetime(df['author_createdAt'], errors='coerce', utc=True)

# Limpiar campos de texto
def limpiar_texto(texto):
    if pd.isna(texto):
        return ""
    return str(texto).replace('\n', ' ').replace('\r', ' ').replace('"', "'")

campos_texto = ['text', 'source', 'author_name', 'author_location', 'place_full_name']
for campo in campos_texto:
    df[campo] = df[campo].apply(limpiar_texto)

# Exportar
df.to_csv('/content/drive/My Drive/DATASET/DATASET_ESTRUCTURADO.csv', index=False, encoding='utf-8', quoting=1)

# Descargar en Colab
from google.colab import files
files.download('/content/drive/My Drive/DATASET/DATASET_ESTRUCTURADO.csv')


/tmp/ipython-input-14-2267491801.py:2: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['createdat'] = pd.to_datetime(df['createdat'], errors='coerce', utc=True)
/tmp/ipython-input-14-2267491801.py:3: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['author_createdAt'] = pd.to_datetime(df['author_createdAt'], errors='coerce', utc=True)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
print("Columnas detectadas:", len(df.columns))
print(df.columns.tolist())

Columnas detectadas: 22
['id', 'url', 'text', 'source', 'createdat', 'author_username', 'author_name', 'author_location', 'author_createdAt', 'author_favouritesCount', 'author_mediaCount', 'author_statusesCount', 'place_coordinates', 'place_type', 'place_country', 'place_full_name', 'place_name', 'place_id', 'place_place_type', 'Unnamed: 19', 'Unnamed: 20', 'Unnamed: 21']


In [ ]:
import pandas as pd
import ast

# Rutas de entrada y salida (ajústalas si estás en Google Colab)
archivo_entrada = '/content/drive/My Drive/DATASET/DATASET_SIN_DUPLICADOS.csv'
archivo_salida = '/content/drive/My Drive/DATASET/DATASET_CORREGIDO_2_WKT.csv'

# Leer el archivo con codificación correcta para conservar emojis
df = pd.read_csv(archivo_entrada, encoding='utf-8')

# Función para convertir de coordenadas tipo GeoJSON a WKT (PostGIS-friendly)
def geojson_to_wkt(geojson_str):
    try:
        if pd.isna(geojson_str) or not isinstance(geojson_str, str):
            return None
        coords = ast.literal_eval(geojson_str)
        if not coords or not isinstance(coords, list):
            return None
        anillo = coords[0]
        if not isinstance(anillo, list) or not all(isinstance(p, list) and len(p) == 2 for p in anillo):
            return None
        puntos = ', '.join([f"{lon} {lat}" for lon, lat in anillo])
        # Cerrar el anillo si es necesario
        if anillo[0] != anillo[-1]:
            puntos += f", {anillo[0][0]} {anillo[0][1]}"
        return f"POLYGON(({puntos}))"
    except Exception as e:
        print(f"Error en coordenadas: {geojson_str} - {e}")
        return None

# Aplicar la conversión a la columna correspondiente
df['place_coordinates'] = df['place_coordinates'].apply(geojson_to_wkt)

# Guardar el CSV corregido con codificación UTF-8 (con emojis)
df.to_csv(archivo_salida, index=False, encoding='utf-8')

print(f"✅ Archivo guardado correctamente como: {archivo_salida}")


Se truncaron las últimas líneas 5000 del resultado de transmisión.
Error en coordenadas: POLYGON((-99.42842 23.3895561, -99.42842 23.9823121, -98.947388 23.9823121, -98.947388 23.3895561, -99.42842 23.3895561)) - invalid syntax. Perhaps you forgot a comma? (<unknown>, line 1)
Error en coordenadas: POLYGON((-93.248341 17.7035, -93.248341 18.340419, -92.584658 18.340419, -92.584658 17.7035, -93.248341 17.7035)) - invalid syntax. Perhaps you forgot a comma? (<unknown>, line 1)
Error en coordenadas: POLYGON((-93.248341 17.7035, -93.248341 18.340419, -92.584658 18.340419, -92.584658 17.7035, -93.248341 17.7035)) - invalid syntax. Perhaps you forgot a comma? (<unknown>, line 1)
Error en coordenadas: POLYGON((-103.4086326 20.6005851, -103.4086326 20.7527191, -103.263131 20.7527191, -103.263131 20.6005851, -103.4086326 20.6005851)) - invalid syntax. Perhaps you forgot a comma? (<unknown>, line 1)
Error en coordenadas: POLYGON((-103.4086326 20.6005851, -103.4086326 20.7527191, -103.263131 20.75

In [ ]:
import pandas as pd

# Ruta de entrada y salida (ajústalas si usas Google Drive)
archivo_entrada = '/content/drive/MyDrive/DATASET/DATASET_FILTRADO.csv'
archivo_salida = '/content/drive/MyDrive/DATASET/DATASET_SIN_DUPLICADOS.csv'

# Cargar el CSV
df = pd.read_csv(archivo_entrada, encoding='utf-8')

# Verifica antes cuántas filas tienes
print(f"Filas originales: {len(df)}")

# OPCIÓN 1: Eliminar duplicados basados en todas las columnas
df_sin_duplicados = df.drop_duplicates()

# OPCIÓN 2 (más segura): Eliminar duplicados por una o más columnas clave
# df_sin_duplicados = df.drop_duplicates(subset=['id', 'text'])

# Guardar CSV limpio
df_sin_duplicados.to_csv(archivo_salida, index=False, encoding='utf-8')

# Reporte final
print(f"Filas sin duplicados: {len(df_sin_duplicados)}")
print(f"✅ Archivo guardado como: {archivo_salida}")


/tmp/ipython-input-7-1691725309.py:8: DtypeWarning: Columns (1,2,3,4,5,6,7,8,12,13,14,15,16,17) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(archivo_entrada, encoding='utf-8')


Filas originales: 1048499
Filas sin duplicados: 237930
✅ Archivo guardado como: /content/drive/MyDrive/DATASET/DATASET_SIN_DUPLICADOS.csv


In [ ]:
import pandas as pd
import geopandas as gpd
from shapely import wkt

archivo = "/content/drive/MyDrive/DATASET/DATASET_150.csv"
df = pd.read_csv(archivo, encoding='utf-8')

# Convertir la columna de WKT a geometría
df['geometry'] = df['place_coordinates'].apply(wkt.loads)

# Crear un GeoDataFrame
gdf = gpd.GeoDataFrame(df, geometry='geometry')

# Calcular centroides
gdf['centroide'] = gdf.geometry.centroid

# Separar en columnas de lat/lon si lo necesitas
gdf['centroide_lon'] = gdf['centroide'].x
gdf['centroide_lat'] = gdf['centroide'].y

# Ver resultados
print(gdf[['id', 'place_name', 'centroide_lon', 'centroide_lat']].head())

# Guardar a CSV si quieres
gdf.to_csv("/content/drive/MyDrive/DATASET/DATASET_150_CENTROIDES.csv", index=False)


                    id          place_name  centroide_lon  centroide_lat
0  1000082083853713409           Querétaro    -100.440595      20.715275
1  1000166664892174336       Benito Juárez     -99.161480      19.380613
2  1000178687793336320  San Pedro Pochutla     -96.396036      15.789067
3  1000185709586731008              Mexico    -102.558037      23.625419
4  1000205783001387008       Benito Juárez     -99.161480      19.380613
